# تدريب نموذج الالتهاب الرئوي v4 — EfficientNetB0 + RSNA

**قبل التشغيل:**
1. `Runtime → Change runtime type → T4 GPU`
2. ادخل على صفحة قواعد مسابقة RSNA واضغط **I Understand and Accept** بنفس حساب Kaggle:
   https://www.kaggle.com/competitions/rsna-pneumonia-detection-challenge/rules
3. جهّز ملف `kaggle.json` من Kaggle → Settings → API → **Create New Token**

ثم `Runtime → Run all`. في الخلية 3 سيُطلب منك رفع `kaggle.json`. في النهاية ينزل ملف `pneumonia_model_v4.zip` تلقائيًا — ضعه في Downloads وأخبر Claude.

**البيانات:**
- Kermany (أطفال): train+val → تدريب/تحقق، و **test (624 صورة) لا تُلمس إلا في القياس النهائي**.
- RSNA (بالغين): فقط `Normal` و `Lung Opacity`. تُستبعد `No Lung Opacity / Not Normal` لأنها صور مريضة بأمراض أخرى وليست طبيعية. تقسيم 80/10/10 حسب المريض.

In [ ]:
# 1) Pin Keras to the same version as the local app (3.12) so the saved model loads there
!pip install -q "keras==3.12.0" pydicom kaggle

In [ ]:
# 2) Environment check
import os, re, glob, json, subprocess
import numpy as np, pandas as pd
import tensorflow as tf, keras
print("TF", tf.__version__, "| Keras", keras.__version__)
assert keras.__version__.startswith("3.12"), "Keras غير مطابق — اعمل Runtime > Restart session ثم Run all"
gpus = tf.config.list_physical_devices("GPU")
print("GPU:", gpus)
assert gpus, "فعّل الـ GPU: Runtime > Change runtime type > T4 GPU"

IMG = 224
SEED = 42
INCLUDE_NOT_NORMAL = False   # keep False: 'Not Normal' images are diseased, not healthy
SAVE_TO_DRIVE = False        # True = copy checkpoints/results to Google Drive as well
np.random.seed(SEED); keras.utils.set_random_seed(SEED)

In [ ]:
# 3) Kaggle credentials
from google.colab import files
os.makedirs("/root/.kaggle", exist_ok=True)
if not os.path.exists("/root/.kaggle/kaggle.json"):
    print("ارفع ملف kaggle.json")
    up = files.upload()
    with open("/root/.kaggle/kaggle.json", "wb") as f:
        f.write(up[next(iter(up))])
os.chmod("/root/.kaggle/kaggle.json", 0o600)
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
# 4) Download both datasets (~2 GB + ~4 GB, a few minutes on Colab)
def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:], r.stderr[-2000:])
        raise RuntimeError(f"failed: {cmd}")

if not glob.glob("/content/kermany/**/test/NORMAL", recursive=True):
    sh("kaggle datasets download -d paultimothymooney/chest-xray-pneumonia -p /content/kermany -q")
    sh("unzip -q -o /content/kermany/chest-xray-pneumonia.zip -d /content/kermany")
print("Kermany ✓")

if not os.path.exists("/content/rsna/stage_2_train_images"):
    try:
        sh("kaggle competitions download -c rsna-pneumonia-detection-challenge -p /content/rsna -q")
    except RuntimeError:
        raise RuntimeError("تحميل RSNA فشل — غالبًا لم تقبل قواعد المسابقة. افتح الرابط في أول الصفحة واضغط Accept ثم أعد تشغيل الخلية.")
    sh('cd /content/rsna && unzip -q -o rsna-pneumonia-detection-challenge.zip "stage_2_train_images/*" stage_2_detailed_class_info.csv')
    os.remove("/content/rsna/rsna-pneumonia-detection-challenge.zip")
print("RSNA ✓")

In [ ]:
# 5) Build file lists and splits
from sklearn.model_selection import GroupShuffleSplit, train_test_split

KROOT = sorted((os.path.dirname(os.path.dirname(p))
                for p in glob.glob("/content/kermany/**/test/NORMAL", recursive=True) if "__MACOSX" not in p), key=len)[0]

def kermany_split(split):
    rows = []
    for label, cls in [(0, "NORMAL"), (1, "PNEUMONIA")]:
        for p in sorted(glob.glob(f"{KROOT}/{split}/{cls}/*")):
            if p.lower().endswith((".jpeg", ".jpg", ".png")):
                rows.append((p, label, "kermany"))
    return pd.DataFrame(rows, columns=["path", "label", "source"])

def kermany_patient(p):
    n = os.path.basename(p)
    m = re.match(r"(person\d+)", n) or re.match(r"(.*IM-\d+)", n)
    return m.group(1) if m else n

k_all = pd.concat([kermany_split("train"), kermany_split("val")], ignore_index=True)
k_test = kermany_split("test")
assert len(k_test) == 624, len(k_test)
groups = k_all.path.map(kermany_patient)
tr_idx, va_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED).split(k_all, groups=groups))
k_train, k_val = k_all.iloc[tr_idx], k_all.iloc[va_idx]

info = pd.read_csv("/content/rsna/stage_2_detailed_class_info.csv").drop_duplicates("patientId")
print(info["class"].value_counts(), "\n")
keep = {"Normal": 0, "Lung Opacity": 1}
if INCLUDE_NOT_NORMAL:
    keep["No Lung Opacity / Not Normal"] = 0
r = info[info["class"].isin(keep)].copy()
r = pd.DataFrame({"path": "/content/rsna/stage_2_train_images/" + r.patientId + ".dcm",
                  "label": r["class"].map(keep).values, "source": "rsna"})
r_train, r_rest = train_test_split(r, test_size=0.2, stratify=r.label, random_state=SEED)
r_val, r_test = train_test_split(r_rest, test_size=0.5, stratify=r_rest.label, random_state=SEED)

train_df = pd.concat([k_train, r_train], ignore_index=True).sample(frac=1, random_state=SEED)
val_df = pd.concat([k_val, r_val], ignore_index=True)
for name, d in [("train", train_df), ("val", val_df), ("test_kermany", k_test), ("test_rsna", r_test)]:
    print(f"{name:13s} {len(d):6d}  normal={int((d.label==0).sum()):6d}  pneumonia={int((d.label==1).sum()):6d}")

In [ ]:
# 6) Load + resize every image once into RAM.
#    prep_pil() is THE preprocessing — app.py must do exactly the same at inference.
from PIL import Image
import pydicom
from concurrent.futures import ProcessPoolExecutor

def prep_pil(img):
    img = img.convert("L").resize((IMG, IMG), Image.BILINEAR)
    return np.asarray(img, dtype=np.uint8)

def load_any(p):
    if p.endswith(".dcm"):
        ds = pydicom.dcmread(p)
        a = ds.pixel_array
        if a.dtype != np.uint8:
            a = a.astype(np.float32)
            a = ((a - a.min()) / max(float(a.max() - a.min()), 1e-6) * 255).astype(np.uint8)
        if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
            a = 255 - a
        img = Image.fromarray(a)
    else:
        img = Image.open(p)
    return prep_pil(img)

def load_all(df):
    with ProcessPoolExecutor(max_workers=os.cpu_count()) as ex:
        X = np.stack(list(ex.map(load_any, df.path.tolist(), chunksize=64)))
    return X, df.label.values.astype(np.float32)

X_train, y_train = load_all(train_df)
X_val, y_val = load_all(val_df)
X_kt, y_kt = load_all(k_test)
X_rt, y_rt = load_all(r_test)
print(X_train.shape, X_val.shape, X_kt.shape, X_rt.shape)

In [ ]:
# 7) tf.data pipelines. Pixels stay in 0..255: EfficientNet rescales internally.
BATCH = 32
aug = keras.Sequential([
    keras.layers.RandomRotation(0.03),          # ~±10°
    keras.layers.RandomZoom(0.1),
    keras.layers.RandomTranslation(0.08, 0.08),
    keras.layers.RandomContrast(0.1),
])  # no horizontal flip: it would move the heart to the wrong side

def to_rgb(x, y):
    x = tf.cast(x, tf.float32)[..., None]
    return tf.repeat(x, 3, axis=-1), y

def make_ds(X, y, train=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if train:
        ds = ds.shuffle(8192, seed=SEED)
    ds = ds.batch(BATCH).map(to_rgb, num_parallel_calls=tf.data.AUTOTUNE)
    if train:
        ds = ds.map(lambda x, y: (aug(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

ds_train = make_ds(X_train, y_train, train=True)
ds_val = make_ds(X_val, y_val)

n_pos = y_train.sum(); n_neg = len(y_train) - n_pos
class_weight = {0: len(y_train) / (2 * n_neg), 1: len(y_train) / (2 * n_pos)}
print("class_weight:", class_weight)

In [ ]:
# 8) Model: EfficientNetB0 pretrained on ImageNet
base = keras.applications.EfficientNetB0(include_top=False, weights="imagenet",
                                         input_shape=(IMG, IMG, 3), pooling="avg")
inp = keras.Input((IMG, IMG, 3), name="xray")
x = base(inp, training=False)       # BatchNorm stays in inference mode during fine-tuning
x = keras.layers.Dropout(0.3)(x)
out = keras.layers.Dense(1, activation="sigmoid", name="pneumonia_prob")(x)
model = keras.Model(inp, out, name="pneumonia_efficientnetb0_v4")

def compile_model(lr):
    model.compile(optimizer=keras.optimizers.Adam(lr), loss="binary_crossentropy",
                  metrics=["accuracy", keras.metrics.AUC(name="auc"),
                           keras.metrics.Precision(name="precision"), keras.metrics.Recall(name="recall")])

# Phase 1 — train only the new head
base.trainable = False
compile_model(1e-3)
h1 = model.fit(ds_train, validation_data=ds_val, epochs=4, class_weight=class_weight)

In [ ]:
# 9) Phase 2 — fine-tune the whole network with a small learning rate
base.trainable = True
compile_model(1e-4)
os.makedirs("/content/export", exist_ok=True)
ckpt = "/content/export/best_v4.keras"
callbacks = [
    keras.callbacks.ModelCheckpoint(ckpt, monitor="val_auc", mode="max", save_best_only=True),
    keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=4, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_auc", mode="max", factor=0.3, patience=2, min_lr=1e-7),
]
h2 = model.fit(ds_train, validation_data=ds_val, epochs=20, class_weight=class_weight, callbacks=callbacks)
model = keras.models.load_model(ckpt)

In [ ]:
# 10) Pick the decision threshold on VALIDATION only (never on test)
from sklearn.metrics import roc_auc_score
def predict(X):
    return model.predict(make_ds(X, np.zeros(len(X), np.float32)), verbose=0).ravel()

def metrics(y, p, th):
    pred = (p > th).astype(int); y = y.astype(int)
    tp = int(((pred == 1) & (y == 1)).sum()); tn = int(((pred == 0) & (y == 0)).sum())
    fp = int(((pred == 1) & (y == 0)).sum()); fn = int(((pred == 0) & (y == 1)).sum())
    rec = tp / max(tp + fn, 1); spec = tn / max(tn + fp, 1); prec = tp / max(tp + fp, 1)
    return {"threshold": round(float(th), 2), "accuracy": round((tp + tn) / len(y) * 100, 2),
            "precision": round(prec * 100, 2), "recall": round(rec * 100, 2),
            "specificity": round(spec * 100, 2), "f1": round(2 * prec * rec / max(prec + rec, 1e-9) * 100, 2),
            "auc": round(float(roc_auc_score(y, p)), 4), "tp": tp, "tn": tn, "fp": fp, "fn": fn}

p_val = predict(X_val)
grid = np.round(np.arange(0.05, 0.96, 0.01), 2)
bal = [(metrics(y_val, p_val, t)["recall"] + metrics(y_val, p_val, t)["specificity"]) / 2 for t in grid]
THRESHOLD = float(grid[int(np.argmax(bal))])
print("Threshold chosen on validation:", THRESHOLD)
print("val @0.5 ", metrics(y_val, p_val, 0.5))
print("val @thr ", metrics(y_val, p_val, THRESHOLD))

In [ ]:
# 11) FINAL test — first and only time the test sets are used
p_kt, p_rt = predict(X_kt), predict(X_rt)
results = {
    "kermany_test_624": {"at_0.5": metrics(y_kt, p_kt, 0.5), "at_threshold": metrics(y_kt, p_kt, THRESHOLD)},
    "rsna_test": {"at_0.5": metrics(y_rt, p_rt, 0.5), "at_threshold": metrics(y_rt, p_rt, THRESHOLD)},
}
for k, v in results.items():
    for t, m in v.items():
        print(f"{k:18s} {t:13s} acc {m['accuracy']:6.2f} | prec {m['precision']:6.2f} | rec {m['recall']:6.2f} "
              f"| spec {m['specificity']:6.2f} | AUC {m['auc']:.4f} | FP {m['fp']} FN {m['fn']}")

In [ ]:
# 12) Export: model + weights + config, then download the zip
model.save("/content/export/pneumonia_model_v4.keras")
model.save_weights("/content/export/pneumonia_model_v4.weights.h5")
os.remove(ckpt)
config = {
    "model": "EfficientNetB0 (ImageNet) fine-tuned",
    "input": {"size": [IMG, IMG], "pipeline": "PIL open -> convert('L') -> resize BILINEAR -> repeat to 3 channels",
              "pixel_range": "0..255 float32 (do NOT divide by 255; model rescales internally)"},
    "output": "sigmoid = P(pneumonia)",
    "threshold": THRESHOLD,
    "trained_on": {"train": len(train_df), "val": len(val_df),
                   "sources": "Kermany chest_xray (train+val) + RSNA Pneumonia (Normal / Lung Opacity)"},
    "test_results": results,
    "versions": {"tensorflow": tf.__version__, "keras": keras.__version__},
}
with open("/content/export/model_config.json", "w") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

sh("cd /content/export && zip -q -r /content/pneumonia_model_v4.zip .")
if SAVE_TO_DRIVE:
    sh("cp /content/pneumonia_model_v4.zip /content/drive/MyDrive/")
files.download("/content/pneumonia_model_v4.zip")